In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import re
import string
import ast

from transformers import pipeline
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
import nltk

# Descargar recursos necesarios
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# ----------- 1. Limpieza de texto para sentimiento ----------
def limpiar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = texto.lower()
    texto = re.sub(f'[{re.escape(string.punctuation)}]', '', texto)
    stop_words = set(stopwords.words('english'))
    stemmer = SnowballStemmer('english')
    palabras = texto.split()
    palabras_limpias = [stemmer.stem(palabra) for palabra in palabras if palabra not in stop_words]
    return ' '.join(palabras_limpias)

# ----------- 2. Cargar archivo ----------
ruta_parquet = r'C:\Users\gonza\OneDrive\Desktop\Proyecto Grupal Henry\archivos_ETL_finales\Maps_review_sitios.parquet'
df = pd.read_parquet(ruta_parquet)
df = df[df['text'].notnull() & df['text'].str.strip().ne("")].copy()

# ----------- 3. Modelo de emociones ----------
print("Cargando modelo de emociones...")
emotion_pipeline = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-emotion"
)

def analizar_emociones(texto, max_chars=400):
    try:
        if not isinstance(texto, str):
            return None
        texto = texto.strip()
        if len(texto) > max_chars:
            texto = texto[:max_chars]
        return emotion_pipeline(texto)
    except Exception as e:
        print(f"Error analizando emociones para texto: {texto[:50]}... -> {e}")
        return None

# Aplicar emociones
print("Aplicando análisis de emociones...")
df['emotions_all'] = df['text'].apply(lambda t: analizar_emociones(t))
df['emotions_all'] = df['emotions_all'].apply(lambda e: sorted(e, key=lambda x: x['score'], reverse=True) if e else None)

df['primary_emotion'] = df['emotions_all'].apply(lambda emociones: emociones[0]['label'] if emociones else None)
df['primary_emotion_score'] = df['emotions_all'].apply(lambda emociones: emociones[0]['score'] if emociones else None)

# ----------- 4. Modelo de sentimiento (cargado localmente) ----------
modelo_sentimiento_path = 'modelo_sentimiento_v1.pkl'
vectorizador_sentimiento_path = 'vectorizador_tfidf_v1.pkl'

try:
    modelo_sentimiento = joblib.load(modelo_sentimiento_path)
    vectorizador_sentimiento = joblib.load(vectorizador_sentimiento_path)
    print("Modelo de sentimiento y vectorizador cargados.")
except FileNotFoundError:
    print("ERROR: No se encontraron los archivos del modelo de sentimiento.")
    modelo_sentimiento = None
    vectorizador_sentimiento = None

# Aplicar análisis de sentimiento en batch
if modelo_sentimiento and vectorizador_sentimiento:
    print("Aplicando análisis de sentimiento...")
    textos_limpios = df['text'].apply(limpiar_texto)
    vectores = vectorizador_sentimiento.transform(textos_limpios)
    df['sentiment'] = modelo_sentimiento.predict(vectores)
else:
    df['sentiment'] = None

# ----------- 5. Gráficos ----------
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='primary_emotion', order=df['primary_emotion'].value_counts().index, palette='Set2')
plt.title('Distribución de emociones principales')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

if df['sentiment'].notnull().sum() > 0:
    plt.figure(figsize=(10, 5))
    sns.countplot(data=df, x='sentiment', order=df['sentiment'].value_counts().index, palette='Set3')
    plt.title('Distribución de sentimientos')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 6))
    sns.countplot(data=df, x='primary_emotion', hue='sentiment', palette='pastel')
    plt.title('Emociones principales según sentimiento')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# ----------- 6. Guardar en Excel ----------
columnas_salida = ['business_name', 'text', 'primary_emotion', 'primary_emotion_score', 'emotions_all', 'sentiment']
ruta_excel = r'C:\Users\gonza\OneDrive\Desktop\emociones_sentimientos_final.xlsx'
df[columnas_salida].to_excel(ruta_excel, index=False)
print(f"\n✅ Archivo Excel guardado en: {ruta_excel}")

# ----------- 7. Distribución de emociones por negocio (normalizada) ----------
print("Generando distribución de emociones por negocio...")

df = df[df['emotions_all'].notnull()].copy()
df['emotions_all'] = df['emotions_all'].apply(ast.literal_eval)

expanded_rows = []
for _, row in df.iterrows():
    for emotion in row['emotions_all']:
        expanded_rows.append({
            'business_name': row['business_name'],
            'emotion': emotion['label'],
            'score': emotion['score']
        })

emotion_df = pd.DataFrame(expanded_rows)

emotion_avg = emotion_df.groupby(['business_name', 'emotion'])['score'].mean().reset_index()
emotion_avg['normalized_score'] = emotion_avg.groupby('business_name')['score'].transform(lambda x: x / x.sum())

emotion_pivot = emotion_avg.pivot(index='business_name', columns='emotion', values='normalized_score').reset_index()

ruta_emociones_normalizadas = r'C:\Users\gonza\OneDrive\Desktop\distribucion_emociones_por_negocio.xlsx'
emotion_pivot.to_excel(ruta_emociones_normalizadas, index=False)
print(f"✅ Archivo de distribución de emociones guardado en: {ruta_emociones_normalizadas}")
